# LookTwice — 03 Evaluasi akhir & export (Kaggle)

Jalankan **setelah** model final dipilih dari hasil validasi (stage 1–3). Test set dipakai di sini saja.

**Add data:** `looktwice-faces`, kode repo, dan output notebook 02 (folder `runs/`).  **Accelerator:** GPU.

In [ ]:
import os, subprocess
REPO_URL = 'https://github.com/garrentanavaro/look-twice.git'; REPO_DATASET = ''
WORK = '/kaggle/working/look-twice'
if not os.path.exists(WORK):
    if REPO_URL: subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, WORK], check=True)
    else: subprocess.run(['cp', '-r', REPO_DATASET, WORK], check=True)
os.chdir(WORK)
!pip install -q -e . timm "albumentations>=2.0" onnx onnxruntime

In [ ]:
import glob
INDEX = sorted(glob.glob('/kaggle/input/**/crops_index.csv', recursive=True))[0]
RUNS = sorted(glob.glob('/kaggle/input/**/runs', recursive=True))[0]
FINAL = 's1_xception_B'
COMPARE = ['s1_resnet50_B', 's1_xception_B', 's1_efficientnet_b4_B']
print(INDEX, RUNS, sep='\n')

## 1. Perbandingan arsitektur di test set
ROC, metrik frame & video, confusion matrix.

In [ ]:
runs = ' '.join(f'--run {RUNS}/{r}' for r in COMPARE)
!python -m looktwice.evaluate $runs --index $INDEX --out results/eval

## 2. Analisis mendalam model final
Robustness (JPEG/blur/downscale), pola error, Grad-CAM.

In [ ]:
!python -m looktwice.evaluate --run $RUNS/$FINAL --index $INDEX --out results/eval_final --robustness --errors --gradcam 6

In [ ]:
from IPython.display import Image, display
import pandas as pd
for f in ['history', 'confusion_frame', 'confusion_video', 'robustness', 'false_positives', 'false_negatives', 'gradcam']:
    print(f); display(Image(f'results/eval_final/{FINAL}/{f}.png'))
for g in ['scene', 'target_actor', 'face_size', 'detector_conf']:
    print(); print('error rate by', g); display(pd.read_csv(f'results/eval_final/{FINAL}/errors_by_{g}.csv'))

## 3. Export ONNX + latency (GPU Kaggle & CPU Kaggle)
Ulangi `--bench` di laptop untuk angka CPU "perangkat pengguna".

In [ ]:
!python -m looktwice.export_onnx --run $RUNS/$FINAL --out weights/looktwice.onnx --bench
pd.read_csv('results/latency.csv')

## 4. Simpan
Download `results/` (di-commit ke repo), `weights/looktwice.onnx` + `.json` (untuk backend), dan `best.pt` run final (untuk benchmark di laptop).

In [ ]:
!mkdir -p /kaggle/working/export && cp -r results weights /kaggle/working/export/ && cp $RUNS/$FINAL/best.pt /kaggle/working/export/
!du -sh /kaggle/working/export/*